In [1]:
import xarray as xr
import os
import glob
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.stats import linregress
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.ticker import MaxNLocator

In [2]:
path='/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files'
pattern=f'ROS_Monthly*.nc'
file_list = sorted(glob.glob(os.path.join(path, pattern)))
data = xr.open_mfdataset(file_list, combine='by_coords',engine='netcdf4')

In [3]:
data_seasonal = data.sum(dim="month")
data_seasonal

<xarray.Dataset> Size: 1GB
Dimensions:              (season: 73, south_north: 450, west_east: 420,
                          interp_level: 3)
Coordinates:
  * season               (season) <U9 3kB '1950-1951' ... '2022-2023'
    XLAT                 (south_north, west_east) float32 756kB dask.array<chunksize=(450, 420), meta=np.ndarray>
    XLONG                (south_north, west_east) float32 756kB dask.array<chunksize=(450, 420), meta=np.ndarray>
  * interp_level         (interp_level) float64 24B 850.0 925.0 950.0
Dimensions without coordinates: south_north, west_east
Data variables: (12/13)
    ros_tally            (season, south_north, west_east) int64 110MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    ros_counts           (season, south_north, west_east) int64 110MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    ros_days_count       (season, south_north, west_east) int64 110MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    rain_sum             (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    rain_ros_sum         (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    rain_avg             (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    ...                   ...
    swe_avg              (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    swe_ros_avg          (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    T2_avg               (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    T2_ros_avg           (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    temp_levels_avg      (season, interp_level, south_north, west_east) float32 166MB dask.array<chunksize=(1, 2, 450, 420), meta=np.ndarray>
    temp_levels_ros_avg  (season, interp_level, south_north, west_east) float32 166MB dask.array<chunksize=(1, 2, 450, 420), meta=np.ndarray>

In [4]:
shapefile_path = "/center1/DYNDOWN/phutton5/ROS/boundaries/Alaska_Borough_and_Census_Area_Boundaries.shp"
borough_boundaries = gpd.read_file(shapefile_path)
borough_boundaries = borough_boundaries.set_crs(epsg=3338)
borough_boundaries = borough_boundaries.to_crs(epsg=4326)
FNSB_boundary = borough_boundaries[borough_boundaries['CommunityN'] == 'Fairbanks North Star Borough']
FNSB_geom = FNSB_boundary.geometry.iloc[0] 
FNSB_coords = []
FNSB_coords.extend(list(FNSB_geom.exterior.coords))
FNSB_coords = np.array(FNSB_coords)  
FNSB_coords = pd.DataFrame({
    "lon": FNSB_coords[:, 0],
    "lat": FNSB_coords[:, 1]})

Fairbanks_lat=(64.84)
Fairbanks_lon=(-147.72)
lat=data['XLAT']
lon=data['XLONG']
seasons=data['season']

In [5]:
seasonslist = seasons.values.tolist()

In [6]:
'''
#joshes code
def new_linregress(x, y):
    # Wrapper around scipy linregress to use in apply_ufunc
    try:
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y, alternative='two-sided')
        return np.array([slope, intercept, r_value, p_value, std_err])
    except: return np.array([np.nan, np.nan, np.nan, np.nan, np.nan])

def xr_linreg(x, y, dim):
    res = xr.apply_ufunc(new_linregress, x, y,
                         input_core_dims=[[dim], [dim]],
                         output_core_dims=[['params']],
                         vectorize=True,
                         dask="parallelized",
                         output_dtypes=['float64'],
                         output_sizes={'params': 5})
    ds_out = res.to_dataset(dim='params').rename({0: 'slope',
                                                  1: 'intercept',
                                                  2: 'rval',
                                                  3: 'pval',
                                                  4: 'stderr'})
    return ds_out
def xr_linreg(x, y, dim):
    res = xr.apply_ufunc(
        new_linregress, x, y,
        input_core_dims=[[dim], [dim]],
        output_core_dims=[['params']],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={'params': 5},
    )

    res = res.assign_coords(
        params=['slope', 'intercept', 'rval', 'pval', 'stderr']
    )

    return res.to_dataset(dim='params')
'''
'''
# Numeric representation of seasons
x_numeric = xr.DataArray(
    np.arange(len(data['season'])),  # 0,1,2,...,72
    dims=['season'],
    coords={'season': data['season']})
ds_trend_2=xr_linreg(x_numeric,data['ros_tally'],dim='season').compute()
#ds_trend = xr_linreg(x_numeric, data['ros_tally'], dim='season').compute()
#took 14 min to run 
'''

"\n# Numeric representation of seasons\nx_numeric = xr.DataArray(\n    np.arange(len(data['season'])),  # 0,1,2,...,72\n    dims=['season'],\n    coords={'season': data['season']})\nds_trend_2=xr_linreg(x_numeric,data['ros_tally'],dim='season').compute()\n#ds_trend = xr_linreg(x_numeric, data['ros_tally'], dim='season').compute()\n#took 14 min to run \n"

In [10]:
x_numeric = xr.DataArray(
    np.arange(len(data['season'])),  # 0,1,2,...,72
    dims=['season'],
    coords={'season': data['season']})

In [7]:
import numpy as np
import xarray as xr
from scipy import stats


def new_linregress(x, y):
    """
    Wrapper around scipy.stats.linregress for use with xarray.apply_ufunc
    """
    try:
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        return np.array([slope, intercept, r_value, p_value, std_err])
    except Exception:
        return np.full(5, np.nan)


def xr_linreg(x, y, dim):
    """
    Apply linear regression along dimension `dim`
    Returns an xarray.Dataset with variables:
    slope, intercept, rval, pval, stderr
    """

    # Ensure core dimension is a single Dask chunk
    if x.chunks is not None:
        x = x.chunk({dim: -1})
    if y.chunks is not None:
        y = y.chunk({dim: -1})

    res = xr.apply_ufunc(
        new_linregress,
        x, y,
        input_core_dims=[[dim], [dim]],
        output_core_dims=[['params']],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={'params': 5},
    )

    # Label the params dimension
    res = res.assign_coords(
        params=['slope', 'intercept', 'rval', 'pval', 'stderr']
    )

    # Convert params dimension into variables
    ds_out = res.to_dataset(dim='params')

    return ds_out


In [1]:
# x and y are DataArrays with dimension "season"

#ds = xr_linreg(x_numeric,data['ros_tally'],dim='season')
#ds.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/TEST_trendout_1950-2022_linear.nc")


In [ ]:
#dT = xr_linreg(x_numeric,data['T2_ros_avg'],dim='season')
#dT.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_trend_T2_ros_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2638727/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [ ]:
#dT = xr_linreg(x_numeric,data['T2_avg'],dim='season')
#dT.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_trend_T2_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2638727/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [ ]:
#ddays = xr_linreg(x_numeric,data['ros_days_count'],dim='season')
#ddays.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_trend_ros_days_count_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2638727/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [ ]:
#rain_ros_sum= xr_linreg(x_numeric,data['rain_ros_sum'],dim='season')
#rain_ros_sum.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_trend_rain_ros_sum_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2638727/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [ ]:
#rain_ros_avg= xr_linreg(x_numeric,data['rain_ros_avg'],dim='season')
#rain_ros_avg.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_trend_rain_ros_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2638727/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [11]:
seasonal_ros=np.sum(data['ros_tally'],axis=1)
seasonal_ros_trend=xr_linreg(x_numeric,seasonal_ros,dim='season')
seasonal_ros_trend.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_seasonal_trend_ros_hourly_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_3490416/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [6]:
#swe_avg
swe_avg= xr_linreg(x_numeric,data['swe_avg'],dim='season')
swe_avg.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_trend_swe_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [7]:
swe_ros_avg= xr_linreg(x_numeric,data['swe_ros_avg'],dim='season')
swe_ros_avg.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/linear_trend_swe_ros_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/3958455852.py:30: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


Theilson Trend
-

In [8]:
import numpy as np
import xarray as xr
from scipy import stats


def new_theilsen(x, y):
    """
    Wrapper around scipy.stats.theilslopes for use with xarray.apply_ufunc
    Returns:
    slope, intercept, low_slope, high_slope
    """
    try:
        # Remove NaNs
        mask = np.isfinite(x) & np.isfinite(y)
        if np.sum(mask) < 2:
            return np.full(4, np.nan)

        slope, intercept, low_slope, high_slope = stats.theilslopes(
            y[mask], x[mask]
        )

        return np.array([slope, intercept, low_slope, high_slope])

    except Exception:
        return np.full(4, np.nan)


def xr_theilsen(x, y, dim):
    """
    Apply Theil–Sen regression along dimension `dim`

    Returns an xarray.Dataset with variables:
    slope, intercept, low_slope, high_slope
    """

    # Ensure core dimension is a single Dask chunk
    if x.chunks is not None:
        x = x.chunk({dim: -1})
    if y.chunks is not None:
        y = y.chunk({dim: -1})

    res = xr.apply_ufunc(
        new_theilsen,
        x, y,
        input_core_dims=[[dim], [dim]],
        output_core_dims=[['params']],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={'params': 4},
    )

    # Label parameters
    res = res.assign_coords(
        params=['slope', 'intercept', 'low_slope', 'high_slope']
    )

    # Convert params dimension into variables
    ds_out = res.to_dataset(dim='params')

    return ds_out


In [ ]:
ros = xr_theilsen(x_numeric,data['ros_tally'],dim='season') 
ros.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_ros_1950-2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [10]:
dT = xr_theilsen(x_numeric,data['T2_ros_avg'],dim='season')
dT.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_T2_ros_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [11]:
dT = xr_theilsen(x_numeric,data['T2_avg'],dim='season')
dT.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_T2_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [12]:
ddays = xr_theilsen(x_numeric,data['ros_days_count'],dim='season')
ddays.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_ros_days_count_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [13]:
rain_ros_sum= xr_theilsen(x_numeric,data['rain_ros_sum'],dim='season')
rain_ros_sum.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_rain_ros_sum_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [14]:
rain_ros_avg= xr_theilsen(x_numeric,data['rain_ros_avg'],dim='season')
rain_ros_avg.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_rain_ros_avg_1950_2022.nc")


/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [15]:

swe_avg= xr_theilsen(x_numeric,data['swe_avg'],dim='season')
swe_avg.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_swe_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


In [16]:
swe_ros_avg= xr_theilsen(x_numeric,data['swe_ros_avg'],dim='season')
swe_ros_avg.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_swe_ros_avg_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2524539/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(


Mann Kendall
-

In [9]:
def new_mannkendall(y):
    """
    Mann–Kendall trend test along 1D array

    Returns:
    tau, p_value, S, var_S
    """
    try:
        y = np.asarray(y)
        mask = np.isfinite(y)
        y = y[mask]

        n = len(y)
        if n < 2:
            return np.full(4, np.nan)

        # Kendall tau + p-value
        tau, p_value = stats.kendalltau(np.arange(n), y)

        # Compute S statistic manually
        S = 0
        for k in range(n - 1):
            S += np.sum(np.sign(y[k+1:] - y[k]))

        # Variance of S (no tie correction)
        var_S = (n * (n - 1) * (2 * n + 5)) / 18

        return np.array([tau, p_value, S, var_S])

    except Exception:
        return np.full(4, np.nan)

def xr_mannkendall(y, dim):
    """
    Apply Mann–Kendall test along dimension `dim`

    Returns Dataset with:
    tau, p_value, S, var_S
    """

    # Ensure core dimension is single chunk
    if y.chunks is not None:
        y = y.chunk({dim: -1})

    res = xr.apply_ufunc(
        new_mannkendall,
        y,
        input_core_dims=[[dim]],
        output_core_dims=[['params']],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={'params': 4},
    )

    res = res.assign_coords(
        params=['tau', 'p_value', 'S', 'var_S']
    )

    return res.to_dataset(dim='params')


In [23]:
data_seasonal.data_vars

Data variables:
    ros_tally            (season, south_north, west_east) int64 110MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    ros_counts           (season, south_north, west_east) int64 110MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    ros_days_count       (season, south_north, west_east) int64 110MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    rain_sum             (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    rain_ros_sum         (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    rain_avg             (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    rain_ros_avg         (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndarray>
    swe_avg              (season, south_north, west_east) float32 55MB dask.array<chunksize=(1, 450, 420), meta=np.ndar

In [14]:
ts_list = []
mk_list = []

for var in data_seasonal.data_vars:

    ts = xr_theilsen(x_numeric, data_seasonal[var], dim='season')
    mk = xr_mannkendall(data_seasonal[var], dim='season')

    # Rename all variables inside returned Dataset
    ts = ts.rename({v: f"{var}_{v}_theilsen" for v in ts.data_vars})
    mk = mk.rename({v: f"{var}_{v}_mannkendall" for v in mk.data_vars})

    ts_list.append(ts)
    mk_list.append(mk)

combined = xr.merge(ts_list + mk_list)
combined.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_mannkndall_seasonal_1950_2022.nc")


/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/3494344491.py:45: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/3494344491.py:45: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYN

In [15]:
ts_list = []
mk_list = []

for var in data_seasonal.data_vars:
    ts = xr_theilsen(x_numeric, data[var], dim='season')
    mk = xr_mannkendall(data[var], dim='season')
    # Rename all variables inside returned Dataset
    ts = ts.rename({v: f"{var}_{v}_theilsen" for v in ts.data_vars})
    mk = mk.rename({v: f"{var}_{v}_mannkendall" for v in mk.data_vars})

    ts_list.append(ts)
    mk_list.append(mk)
combined = xr.merge(ts_list + mk_list)
combined.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_mannkndall_monthly_1950_2022.nc")

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/3494344491.py:45: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/2810103515.py:42: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_1585518/3494344491.py:45: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  res = xr.apply_ufunc(
/center1/DYN

NEW COMBINED ON JULY 22
-

In [ ]:

import numpy as np
import xarray as xr
from scipy import stats


# ==========================================================
# Theil-Sen regression
# ==========================================================

def new_theilsen(x, y):
    """
    Theil-Sen trend estimator

    Returns:
    slope,
    intercept,
    low_slope,
    high_slope,
    pct_change
    """

    try:

        mask = np.isfinite(x) & np.isfinite(y)

        if np.sum(mask) < 2:
            return np.full(5, np.nan)

        x = x[mask]
        y = y[mask]

        slope, intercept, low_slope, high_slope = stats.theilslopes(
            y, x
        )

        # fitted start/end values
        start_fit = intercept + slope * x[0]
        end_fit = intercept + slope * x[-1]

        if start_fit != 0:
            pct_change = (
                (end_fit - start_fit) /
                start_fit
            ) * 100
        else:
            pct_change = np.nan


        return np.array([
            slope,
            intercept,
            low_slope,
            high_slope,
            pct_change
        ])


    except Exception:

        return np.full(5, np.nan)


def xr_theilsen(x, y, dim):

    if x.chunks is not None:
        x = x.chunk({dim: -1})

    if y.chunks is not None:
        y = y.chunk({dim: -1})


    res = xr.apply_ufunc(
        new_theilsen,
        x,
        y,
        input_core_dims=[[dim], [dim]],
        output_core_dims=[['params']],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={'params':5},
    )


    res = res.assign_coords(
        params=[
            'slope',
            'intercept',
            'low_slope',
            'high_slope',
            'pct_change'
        ]
    )
    return res.to_dataset(dim='params')


def new_mannkendall(y):
    """
    Mann-Kendall trend test
    Returns:
    tau,
    p_value,
    S,
    var_S
    """
    try:
        y = np.asarray(y)
        mask = np.isfinite(y)
        y = y[mask]
        n = len(y)
        if n < 2:
            return np.full(4, np.nan)
        tau, p_value = stats.kendalltau(
            np.arange(n),
            y)

        # S statistic
        S = 0
        for k in range(n-1):
            S += np.sum(
                np.sign(
                    y[k+1:] - y[k]
                )
            )
        # variance (no tie correction)
        var_S = (
            n *
            (n-1) *
            (2*n+5)
        ) / 18

        return np.array([
            tau,
            p_value,
            S,
            var_S
        ])


    except Exception:

        return np.full(4, np.nan)

def xr_mannkendall(y, dim):

    if y.chunks is not None:
        y = y.chunk({dim:-1})
        
    res = xr.apply_ufunc(
        new_mannkendall,
        y,
        input_core_dims=[[dim]],
        output_core_dims=[['params']],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={'params':4},)


    res = res.assign_coords(
        params=[
            'tau',
            'p_value',
            'S',
            'var_S'
        ])


    return res.to_dataset(dim='params')

In [ ]:
#trend = xr_theilsen(years,precip,dim="time")
#mk = xr_mannkendall(precip,dim="time")
#results = xr.merge([trend,mk])

ts_list = []
mk_list = []

for var in data_seasonal.data_vars:
    ts = xr_theilsen(x_numeric, data[var], dim='season')
    mk = xr_mannkendall(data[var], dim='season')
    # Rename all variables inside returned Dataset
    ts = ts.rename({v: f"{var}_{v}_theilsen" for v in ts.data_vars})
    mk = mk.rename({v: f"{var}_{v}_mannkendall" for v in mk.data_vars})

    ts_list.append(ts)
    mk_list.append(mk)
combined = xr.merge(ts_list + mk_list)
combined.to_netcdf("/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/theilsen_trend_mannkndall_monthly_1950_2022.nc")